In [ ]:
!pip install 'scanpy[leiden]'

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install --quiet scvi-colab
    from scvi_colab import install
    install()
    !pip install --quiet git+https://github.com/BayraktarLab/cell2location#egg=cell2location[tutorials]

In [ ]:
import scanpy as sc
import anndata
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

import cell2location
import scvi

from matplotlib import rcParams
rcParams['pdf.fonttype'] = 42 # enables correct plotting of text for PDFs

In [4]:
# Load ST and single-cell datasets
adata_vis = sc.read(f'/content/drive/MyDrive/st_c2b2_hi_033023.h5ad') # ST dataset
adata_ref = sc.read(f'/content/drive/MyDrive/bbt_rampup_rampdown_final_rna.h5ad') # single-cell datasets

In [ ]:
adata_vis

In [ ]:
adata_ref

In [7]:
"""
Getting the bbt Anndata in the format cell2location expects.
When exporting from Seurat, the raw counts matrix was put in the adata.raw location.
While cell2location (and other tools) expects the raw counts matrix to be in adata.X.
"""
adata_ref = adata_ref.raw.to_adata()
adata_ref.X = adata_ref.X.toarray()
cell2location.models.Cell2location.setup_anndata(adata_ref)

In [8]:
"""
Train the model
"""
cell2location.models.RegressionModel.setup_anndata(adata=adata_ref, batch_key='SampleID', labels_key='Secondary_CellTypes')
from cell2location.models import RegressionModel
mod = RegressionModel(adata_ref)
mod.train(max_epochs=250, accelerator='gpu')

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:

Training:   0%|          | 0/250 [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=250` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=250` reached.


In [9]:
# Output Paths
results_folder = "/content/"
ref_run_name = f'{results_folder}/reference_signatures_b2'
run_name = f'{results_folder}/cell2location_map_b2'

In [ ]:
# Save model
adata_ref = mod.export_posterior(adata_ref, sample_kwargs={'num_samples': 1000, 'batch_size': 2500, 'accelerator': 'gpu'})
mod.save(f"{ref_run_name}", overwrite=True)

In [17]:
adata_file = f"{ref_run_name}/bbt_with_trained_model.h5ad"

adata_ref.var = adata_ref.var.drop(columns=['_index'], errors='ignore')
adata_ref.obs = adata_ref.obs.drop(columns=['_index'], errors='ignore')

adata_ref.write(adata_file)

In [13]:
# Model Training Plots
from matplotlib import pyplot as plt
mod.plot_history(20)
plt.savefig("/content/mod_train.png")
plt.clf()

<Figure size 640x480 with 0 Axes>

In [ ]:
mod.plot_QC()
plt.savefig("/content/mod_train_qc.png")
plt.clf()

In [16]:
# Model Training Outputs
inf_aver = adata_ref.varm['means_per_cluster_mu_fg'][[f'means_per_cluster_mu_fg_{i}'
                                                      for i in adata_ref.uns['mod']['factor_names']]].copy()
inf_aver.columns = adata_ref.uns['mod']['factor_names']
inf_aver.iloc[0:5, 0:5]
inf_aver.to_csv("/content/bbt_b2_reference_signatures.csv")

In [ ]:
"""
Use the model on the Spatial data
"""
# Prepare anndata for cell2location model
adata_vis = adata_vis.raw.to_adata()
adata_vis.X = adata_vis.X.toarray()
adata_vis.var_names = inf_aver.index
cell2location.models.Cell2location.setup_anndata(adata=adata_vis, batch_key="sample")
del adata_vis.var

In [32]:
# Predict
mod_8_200 = cell2location.models.Cell2location(adata_vis, cell_state_df=inf_aver, N_cells_per_location=8, detection_alpha=200) # N_cells_per_location manually estimated
mod_8_200.view_anndata_setup()
mod_8_200.train(max_epochs=30000, batch_size=None, train_size=1, accelerator='gpu')
adata_vis = mod_8_200.export_posterior(adata_vis, sample_kwargs={'num_samples': 1000, 'batch_size': mod_8_200.adata.n_obs, 'accelerator': 'gpu'})

Anndata setup with scvi-tools version 1.4.3.

Setup via `Cell2location.setup_anndata` with arguments:

{
│   'layer': None,
│   'batch_key': 'sample',
│   'labels_key': None,
│   'categorical_covariate_keys': None,
│   'continuous_covariate_keys': None
}

         Summary Statistics         
┏━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓
┃     Summary Stat Key     ┃ Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩
│         n_batch          │   7   │
│         n_cells          │ 3971  │
│ n_extra_categorical_covs │   0   │
│ n_extra_continuous_covs  │   0   │
│         n_labels         │   1   │
│          n_vars          │ 25132 │
└──────────────────────────┴───────┘

               Data Registry                
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Registry Key ┃    scvi-tools Location    ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      X       │          adata.X          │
│    batch     │ adata.obs['_scvi_batch']  │
│    ind_x     │   adata.obs['_indices']   │
│    labels    │ adata.obs['_scvi_labels'] │
└──────────────┴───────────────────────────┘

...

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:

Training:   0%|          | 0/30000 [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=30000` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=30000` reached.


Sampling local variables, batch:   0%|          | 0/1 [00:00<?, ?it/s]

Sampling global variables, sample:   0%|          | 0/999 [00:00<?, ?it/s]

In [33]:
# Output predictions
adata_vis.obsm['q05_cell_abundance_w_sf'].to_csv("/content/cell2location_b2_spatial_output_q05.csv")
adata_vis.obsm['q95_cell_abundance_w_sf'].to_csv("/content/cell2location_b2_spatial_output_q95.csv")
adata_vis.obsm['means_cell_abundance_w_sf'].to_csv("/content/cell2location_b2_spatial_output_means.csv")
adata_vis.obsm['stds_cell_abundance_w_sf'].to_csv("/content/cell2location_b2_spatial_output_stds.csv")

In [34]:
# Save the model itself
mod_8_200.save(f"{run_name}", overwrite=True)
adata_file = f"{run_name}/sp_trained.h5ad"
adata_vis.write(adata_file)